In [1]:
from itertools import product
import os
import pandas as pd
import shutil

In [2]:
# already reduced to presolve < 5000 x 5000
instance_set = "miplib3"
max_runtime = 3600
expected_instances = 2
degrees = [-1, 1]
perturbations = ["objective", "rhs"]

# Get breakdown of successes and failures by mode

In [3]:
# what I want to know is which error modes account for which incomplete test sets

# running list of strings contained by different error codes
# last two are catchalls
err = {
    "skipping": [],
    f"walltime": [],
    "failed to perturb": [],
    "gurobipy.gurobierror: out of memory": [],
    "fileexistserror": [],
    "vmem": [],
}

warn = {
    "samples of" : [],
    "invalid value encountered in double_scalars": [],
    "could not be solved for": [],
    "presolve failed to reduce": [],
}

# runs that weren't run
no_go = []

# runs that errored out with new error code
other = []

# runs that had no errors
empty = []

# count of instances made
count = {}

for i, file_name in enumerate(os.listdir(os.path.join("instances", instance_set))):
    
    # skip anything not a mip
    if not file_name.endswith(".mps"):
        continue
        
    # get the model name
    stem = file_name[:-4]
    
    # instantiate the count
    count[stem] = {}
    for p, d in product(perturbations, degrees):
        if os.path.isdir(os.path.join("test_sets", instance_set, stem, f"{p}_{d}")):
            count[stem][(p, d)] = len([f for f in os.listdir(os.path.join("test_sets", instance_set, stem, f"{p}_{d}")) if f.endswith(".mps")])
        else:
            count[stem][(p, d)] = 0
        
    for degree in degrees:
        
        stem_degree = f"{stem}_{degree}"
        
        # get the error file path
        err_pth = os.path.join("outfiles", f"{stem_degree}.err")
        
        # check if the series wasn't run
        if not os.path.exists(err_pth):
            no_go.append(stem_degree)
        
        # check if the series ran with no errors or warnings
        elif os.path.getsize(err_pth) == 0:
            empty.append(stem_degree)
            
        else:
            # read in file
            with open(err_pth, "r") as f:
                text = f.read().lower()
                
            # check for error codes
            found_code = False
            for code in err:
                if code in text:
                    err[code].append(stem_degree)
                    found_code = True
                    break
            
            if not found_code:
                for code in warn:
                    if code in text:
                        warn[code].append(stem_degree)
                        found_code = True
                        break
                        
            if not found_code:
                other.append(stem_degree)
                
# count of possible folders
expected_folders = len(degrees) * (i + 1)

In [4]:
no_go

[]

In [5]:
# skipping - could not solve base instance in < 1 hour
len(err[f"skipping"]) / expected_folders

0.023809523809523808

In [6]:
err[f"skipping"]

['markshare2_-1', 'mkc_-1', 'mkc_1']

In [7]:
# perturbation algorithm failed to complete due to not making any perturbations
len(err["failed to perturb"]) / expected_folders

0.0

In [8]:
# ran out of memory
bumpable = set()
print(set(err["gurobipy.gurobierror: out of memory"] + err["vmem"]))
print(len(set(err["gurobipy.gurobierror: out of memory"] + err["vmem"])) / expected_folders)
df = pd.read_csv("more_memory.csv", index_col=0)
for file_degree_name in set(err["gurobipy.gurobierror: out of memory"] + err["vmem"]):
    file_name = file_degree_name.rsplit("_")[0] + ".mps"
    current_memory = df["memory"].get(file_name, 4)  # df.loc[file_name, "memory"]
    if current_memory < 15 and file_name not in bumpable:
        bumpable.add(file_name)
        df.loc[file_name, "memory"] = min(15, current_memory * 2)
print(len(bumpable) / expected_folders)
df["memory"].astype(int)
df.to_csv("more_memory.csv")
for file_name in bumpable:
    if os.path.isdir(os.path.join("test_sets", instance_set, file_name[:-4])):
        # shutil.rmtree(os.path.join("test_sets", instance_set, file_name[:-4]))
        print("removed: ", file_name[:-4])

set()
0.0
0.0


In [9]:
# less of an issue - perturbation algorithm failed to complete due to time
len(err[f"walltime"]) / expected_folders

0.0

In [10]:
# file exists error - these just need rerun
len(err["fileexistserror"]) / expected_folders

0.0

In [11]:
err["fileexistserror"]

[]

In [12]:
# less of an issue - perturbation algorithm failed to complete due to exhaustion of perturbation attempts
len(warn["samples of"]) / expected_folders

0.21428571428571427

In [13]:
# nonissue
len(warn["invalid value encountered in double_scalars"]) / expected_folders

0.0

In [14]:
# nonissue
len(warn["could not be solved for"]) / expected_folders

0.0

In [15]:
# nonissue
len(warn["presolve failed to reduce"]) / expected_folders

0.09523809523809523

In [16]:
len(empty) / expected_folders

0.6507936507936508

In [17]:
print(other)
len(other) / expected_folders

['markshare2_1', 'pk1_-1']


0.015873015873015872

In [18]:
empty

['seymour_-1',
 'seymour_1',
 'noswot_-1',
 'noswot_1',
 'rout_-1',
 'p0201_-1',
 'stein45_-1',
 'stein45_1',
 'misc03_-1',
 'misc03_1',
 'misc07_-1',
 'misc07_1',
 'misc06_-1',
 'misc06_1',
 'mod008_-1',
 'mod008_1',
 'fixnet6_-1',
 'fixnet6_1',
 'p0548_-1',
 'p0548_1',
 'gesa3_o_-1',
 'markshare1_-1',
 'markshare1_1',
 'enigma_1',
 '10teams_-1',
 '10teams_1',
 'vpm2_-1',
 'bell5_-1',
 'bell5_1',
 'stein27_-1',
 'stein27_1',
 'dcmulti_-1',
 'dcmulti_1',
 'vpm1_-1',
 'pk1_1',
 'arki001_-1',
 'gesa2_o_-1',
 'bell3a_-1',
 'bell3a_1',
 'harp2_-1',
 'harp2_1',
 'gen_-1',
 'p0282_-1',
 'p0282_1',
 'lseu_-1',
 'lseu_1',
 'gesa2_-1',
 'gesa3_-1',
 'l152lav_-1',
 'l152lav_1',
 'cap6000_-1',
 'egout_-1',
 'egout_1',
 'mas76_-1',
 'mas76_1',
 'rgn_-1',
 'rgn_1',
 'mas74_-1',
 'mas74_1',
 'qnet1_-1',
 'mod010_-1',
 'mod010_1',
 'p2756_-1',
 'gt2_-1',
 'gt2_1',
 'khb05250_-1',
 'khb05250_1',
 'dsbmip_-1',
 'pp08aCUTS_-1',
 'pp08aCUTS_1',
 'qiu_-1',
 'danoint_-1',
 'modglob_-1',
 'modglob_1',
 'set

In [19]:
complete = []
removals = []
for stem in count:
    if len([count for (p, d), count in count[stem].items() if count < expected_instances]) == 0:
        complete.append(stem)
    else:
        # remove the directory
        if os.path.isdir(os.path.join("test_sets", instance_set, stem)):
            # shutil.rmtree(os.path.join("test_sets", instance_set, stem))
            removals.append(stem)
len(complete) / expected_folders

0.2619047619047619

In [20]:
len(removals) / expected_folders

0.16666666666666666

In [21]:
for stem in removals:
    for (p, d), amt in count[stem].items():
        if amt < expected_instances:
            print(stem, p, d, amt)

seymour objective -1 1
rout rhs 1 0
p0201 rhs 1 0
qnet1_o rhs -1 0
qnet1_o rhs 1 0
gesa3_o rhs 1 0
markshare1 objective -1 1
markshare1 objective 1 1
enigma rhs -1 0
vpm2 rhs 1 0
blend2 rhs -1 0
blend2 rhs 1 0
vpm1 rhs 1 0
arki001 rhs 1 0
gesa2_o rhs 1 0
gen rhs 1 0
gesa2 rhs 1 0
gesa3 rhs 1 0
cap6000 rhs 1 0
rentacar rhs 1 0
p2756 rhs 1 0
dsbmip rhs 1 0
danoint rhs 1 0
fiber rhs 1 0
